In [ ]:
import numpy as np

# ❌ resolve fctns manquantes


# La boite générique

In [ ]:
class Boite():

  def __init__(self):
    pass

  def forward(self, inputs):
    self.inputs = inputs
    self.output = self.operation()
    return self.output

  def backward(self, derivee_output):
    assert derivee_output.shape == self.output.shape, f"La dérivée_output reçue a un shape {derivee_output.shape} et différent du shape de output : {self.output.shape}"

    self.derivee_inputs = self.gradient(derivee_output)
    assert self.derivee_inputs.shape == self.inputs.shape, f"La dérivée_input calculée a un shape {self.derivee_inputs.shape } et différent du shape de inputs : {self.inputs.shape}"

    return self.derivee_inputs


  def operation(self):
    pass

  def gradient(self, derivee_output):
    pass


# La boite paramètrée

In [62]:
class BoiteParam():

  def __init__(self, param):
    self.param = param

  def forward(self, inputs):
    self.inputs = inputs
    self.output = self.operation()
    return self.output

  def backward(self, derivee_output):
    assert derivee_output.shape == self.output.shape, f"La derivee_output reçue a un shape {derivee_output.shape} et different du shape de output : {self.output}"

    self.derivee_inputs = self.gradient(derivee_output)
    assert self.derivee_inputs.shape == self.inputs.shape, f"La derivee_input calculée a un shape {self.derivee_inputs.shape } et different du shape de inputs : {self.inputs.shape}"

    self.derivee_param = self.gradient_param(derivee_output)
    assert self.derivee_param.shape == self.param.shape, f"La derivee de param a un shape {self.derivee_param.shape} et different du shape de param : {self.param.shape}"

    return self.derivee_inputs


  def operation(self):
    pass

  def gradient(self, derivee_output):
    pass


  def gradient_param(self, derivee_output):
    pass


# La classe Dot

In [63]:
class Dot(BoiteParam):

  def __init__(self, weights):
    super().__init__(weights)

  def operation(self):
    return np.dot(self.inputs, self.param)

  def gradient(self, derivee_output):
    return np.dot( derivee_output, self.param.T)

  def gradient_param(self, derivee_output):
    return np.dot(self.inputs.T, derivee_output)

  def __repr__(self):
    return "DotProduct"


In [64]:
X = np.array([[ 2,  3, -2],
       [ 4,  5, -1],
       [-5,  2,  3],
       [ 0,  5,  4]])


In [65]:
X.shape


(4, 3)

In [66]:
W = np.array([[ 0.49671415],
 [-0.1382643 ],
 [ 0.64768854]])
W.shape


(3, 1)

In [67]:
np.dot(X, W)


array([[-0.71674168],
       [ 0.64784656],
       [-0.81703373],
       [ 1.89943266]])

In [68]:
M = Dot(weights=W)
out = M.forward(X)
out


array([[-0.71674168],
       [ 0.64784656],
       [-0.81703373],
       [ 1.89943266]])

In [69]:
M


DotProduct

In [70]:
d_out = np.random.randn(4, 1)
d_out


array([[ 1.52302986],
       [-0.23415337],
       [-0.23413696],
       [ 1.57921282]])

In [71]:
M.backward(d_out)


array([[ 0.75651048, -0.21058066,  0.98644898],
       [-0.11630729,  0.03237505, -0.15165846],
       [-0.11629914,  0.03237278, -0.15164782],
       [ 0.78441735, -0.21834875,  1.02283804]])

# La classe ADD

In [72]:
class Add(BoiteParam):

  def __init__(self, biais):
    super().__init__(biais)

  def operation(self):
    return self.inputs + self.param

  def gradient(self, derivee_output):
    return np.ones_like(self.inputs) * derivee_output

  def gradient_param(self, derivee_output):
    r =  np.ones_like(self.param) * derivee_output
    return r.sum(axis=0).reshape(1, self.param.shape[1])

  def __repr__(self):
    return "AddBiais"


In [73]:

B = np.random.rand(1, 1)
B


array([[0.02058449]])

In [74]:
b = Add(biais=B)
b


AddBiais

In [75]:
out_b = b.forward(out)
out_b


array([[-0.69615719],
       [ 0.66843105],
       [-0.79644924],
       [ 1.92001715]])

In [76]:
d_out


array([[ 1.52302986],
       [-0.23415337],
       [-0.23413696],
       [ 1.57921282]])

In [77]:
b.backward(d_out)


array([[ 1.52302986],
       [-0.23415337],
       [-0.23413696],
       [ 1.57921282]])

# La classe Sigmoid

In [78]:
class Sigmoid(Boite):

  def __init__(self):
    super().__init__()

  def operation(self):
    return 1 / (1 + np.exp(-1 * self.inputs))

  def gradient(self, derivee_output):
    return self.output * (1 - self.output) * derivee_output

  def __repr__(self):
    return "sigmoid"


In [79]:
sig = Sigmoid()


In [80]:
sig_out = sig.forward(out_b)
sig_out


array([[0.33266478],
       [0.66115176],
       [0.31078557],
       [0.87214035]])

In [81]:
sig.backward(d_out)


array([[ 0.33811099],
       [-0.05245741],
       [-0.05015164],
       [ 0.17610049]])

# La classe Loss

In [82]:
class Loss():

  def __init__(self):
    pass

  def forward(self, prediction, target):
    assert prediction.shape == target.shape, f"Prediction shape {prediction.shape}  Target shape {target.shape}"
    self.prediction = prediction
    self.target = target
    loss = np.mean((self.target - self.prediction) ** 2)
    return loss

  def backward(self):

    self.loss_derivee = -2 * (self.target - self.prediction)
    assert self.loss_derivee.shape == self.prediction.shape, f"La derivee du loss un shape {self.loss_derivee.shape } et different du shape de Prediction : {self.prediction.shape}"

    return self.loss_derivee


In [83]:
X


array([[ 2,  3, -2],
       [ 4,  5, -1],
       [-5,  2,  3],
       [ 0,  5,  4]])

In [84]:
Y = np.random.randn(4, 1)
Y


array([[ 0.76743473],
       [-0.58087813],
       [-0.52516981],
       [-0.57138017]])

In [85]:
P = sig_out


In [86]:
mse = Loss()


In [87]:
mse.forward(P, Y)


np.float64(1.1285590070133853)

In [88]:
mse.backward()


array([[-0.8695399 ],
       [ 2.48405978],
       [ 1.67191076],
       [ 2.88704102]])

# La Classe Dense

In [89]:
class Dense():

  def __init__(self, neurons, activation=None):
    self.neurons = neurons
    self.activation = activation
    self.params = []
    self.suite = []
    self.initialisation = True


  def build(self, inputs):
    # weights initialization
    np.random.seed(42)

    self.weights = np.random.randn(inputs.shape[1], self.neurons)
    self.biais = np.random.randn(1, self.neurons)

    self.params.append(self.weights)
    self.params.append(self.biais)

    # construction de la suite d'opération
    self.suite = [Dot(weights=self.params[0]), Add(biais=self.params[1])]
    if self.activation:
      self.suite.append(self.activation)



  def forward(self, inputs):
    if self.initialisation:
      self.build(inputs)
      self.initialisation = False

    for boite in self.suite:
      inputs = boite.forward(inputs)

    self.output = inputs

    return self.output


  def backward(self, derivee_output):
    assert derivee_output.shape == self.output.shape

    for boite in reversed(self.suite):
      derivee_output = boite.backward(derivee_output)

    derivee_inputs = derivee_output

    self.get_layer_gradients()

    return derivee_inputs

  def get_layer_gradients(self):

    self.derivee_params = []

    for boite in self.suite:
      if issubclass(boite.__class__, BoiteParam):
        self.derivee_params.append(boite.derivee_param)



  def __repr__(self):
    r = f"DenseLayer(neurons={self.neurons})"
    if self.activation:
      r += " avec Sigmoid"

    return r




In [90]:
sigmoid = Sigmoid()


In [91]:
couche = Dense(neurons=2, activation=sigmoid)


In [92]:
couche


DenseLayer(neurons=2) avec Sigmoid

In [93]:
X


array([[ 2,  3, -2],
       [ 4,  5, -1],
       [-5,  2,  3],
       [ 0,  5,  4]])

In [94]:
couche.forward(X)


array([[0.99320003, 0.99604286],
       [0.99912347, 0.99968533],
       [0.42276305, 0.97817014],
       [0.97978765, 0.99941659]])

In [95]:
couche.params


[array([[ 0.49671415, -0.1382643 ],
        [ 0.64768854,  1.52302986],
        [-0.23415337, -0.23413696]]),
 array([[1.57921282, 0.76743473]])]

In [96]:
d_out = np.random.randn(4, 2)
d_out


array([[-0.46947439,  0.54256004],
       [-0.46341769, -0.46572975],
       [ 0.24196227, -1.91328024],
       [-1.72491783, -0.56228753]])

In [97]:
couche.backward(d_out)


array([[-0.00187061,  0.00120335,  0.00024173],
       [-0.00018133, -0.00048599,  0.00012933],
       [ 0.03497832, -0.02397904, -0.00426045],
       [-0.0169224 , -0.02262434,  0.00807543]])

In [98]:
couche.suite


[DotProduct, AddBiais, sigmoid]

# La classe Model

In [99]:
from copy import deepcopy


In [ ]:
class Model():

  def __init__(self, layers):
    self.layers = layers
    self.compiled = False


  def forward(self, inputs):


    for layer in self.layers:

      inputs = layer.forward(inputs)
    self.output = inputs
    return self.output


  def backward(self, loss_derivee):

    assert loss_derivee.shape == self.output.shape

    for layer in reversed(self.layers):
      loss_derivee = layer.backward(loss_derivee)

    return None

  def get_params(self):
    for layer in self.layers:
      yield from layer.params


  def get_derivee_params(self):
    for layer in self.layers:
      yield from layer.derivee_params


  def update(self):

    for (param, derivee_param) in zip(self.get_params(), self.get_derivee_params()):
      assert param.shape == derivee_param.shape
      param -=   self.learning_rate * derivee_param


  def compile(self, loss, learning_rate):
    self.loss = loss
    self.learning_rate = learning_rate
    self.compiled = True


  def fit(self, X, Y, epochs, validation_data=None):

    if validation_data:
      assert len(validation_data) == 2
      assert validation_data[0].shape[1] == X.shape[1]
      assert validation_data[1].shape[1] == Y.shape[1]

    self.history = {"loss":[]}
    if validation_data:
      self.history['val_loss'] = []


    if not self.compiled:
      raise NotImplementedError("Pas de loss et de learning_rate: Compilez")

    for epoch in range(epochs):
      # forward pass
      predictions = model.forward(X)
      loss = self.loss.forward(predictions, Y)
      self.history['loss'].append(loss)


      # val loss
      if validation_data:
        val_preds = model.forward(validation_data[0])
        val_loss = self.loss.forward(val_preds, validation_data[1])
        self.history['val_loss'].append(val_loss)

      log = f'Epoch {epoch+1} .............. loss : {loss}'
      if validation_data:
        log += f"  ....val_loss : {val_loss}"
      print(log)

      # backward pass
      loss_derivee = self.loss.backward()
      self.backward(loss_derivee)

      # update
      self.update()

    return self.history

  def save_model(self, file):
    model_save = deepcopy(self)

    import pickle
    with open(file, "wb") as f:
      pickle.dump(model_save, f)

  def __repr__(self):

    r = "Layers ................."
    for layer in self.layers:
      r += f" \n {str(layer)}"

    return r


In [101]:
model = Model(layers = [ Dense(neurons=3, activation=sigmoid),
       Dense(neurons=1)])


In [102]:
model


Layers ................. 
 DenseLayer(neurons=3) avec Sigmoid 
 DenseLayer(neurons=1)

In [103]:
X


array([[ 2,  3, -2],
       [ 4,  5, -1],
       [-5,  2,  3],
       [ 0,  5,  4]])

In [104]:
Y


array([[ 0.76743473],
       [-0.58087813],
       [-0.52516981],
       [-0.57138017]])

In [105]:
model.forward(X)


array([[2.47005633],
       [2.53479834],
       [1.89807891],
       [1.92678186]])

In [106]:
loss_derivee = np.random.randn(4, 1)
loss_derivee


array([[-0.23415337],
       [-0.23413696],
       [ 1.57921282],
       [ 0.76743473]])

In [107]:
model.backward(loss_derivee)


In [108]:
model.get_params()


<generator object Model.get_params at 0x000001FAD12AAA80>

In [109]:
model.get_derivee_params()


<generator object Model.get_derivee_params at 0x000001FAD12AA670>

# Update des paramètres

In [110]:
M


DotProduct

In [111]:
issubclass(M.__class__, BoiteParam)


True

In [112]:
M.__class__


__main__.Dot

In [113]:
M.param


array([[ 0.49671415],
       [-0.1382643 ],
       [ 0.64768854]])

In [114]:
M.derivee_inputs


array([[ 0.75651048, -0.21058066,  0.98644898],
       [-0.11630729,  0.03237505, -0.15165846],
       [-0.11629914,  0.03237278, -0.15164782],
       [ 0.78441735, -0.21834875,  1.02283804]])

In [115]:
couche.params


[array([[ 0.49671415, -0.1382643 ],
        [ 0.64768854,  1.52302986],
        [-0.23415337, -0.23413696]]),
 array([[1.57921282, 0.76743473]])]

In [116]:
couche.derivee_params


[array([[-0.30320043,  0.20796531],
        [-0.06424681, -0.07766607],
        [ 0.04724885, -0.1280065 ]]),
 array([[ 0.02131064, -0.03919074]])]

In [117]:
model = Model(layers = [ Dense(neurons=2, activation=sigmoid),
       Dense(neurons=1)])
#forward
P = model.forward(X)
mse = Loss()
loss = mse.forward(P, Y)
print(loss)
#backward
loss_derivee = mse.backward()
model.backward(loss_derivee)


1.6466935300286027


In [118]:
model.update()


AttributeError: 'Model' object has no attribute 'learning_rate'

In [ ]:
P = model.forward(X)
mse = Loss()
loss = mse.forward(P, Y)
print(loss)


In [ ]:
P


In [ ]:
X


In [ ]:
model.forward(X)


In [ ]:
model.layers[1].params


# Test de la fonction fit

In [ ]:
model = Model(layers = [ Dense(neurons=2, activation=sigmoid),
       Dense(neurons=1)])
mse = Loss()
model.compile(loss=mse, learning_rate=0.01)
h = model.fit(X, Y, epochs=10)


# Comparaison Tensorflow

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import SGD

model = Sequential([Dense(units=2, activation='sigmoid'),
                    Dense(units=1)])
model.compile(optimizer=SGD(learning_rate=0.1), loss='mse')
history = model.fit(X, Y, epochs=10)


In [ ]:
??Sequential


# Validation

In [ ]:
model = Model(layers = [ Dense(neurons=2, activation=sigmoid),
       Dense(neurons=1)])
mse = Loss()
model.compile(loss=mse, learning_rate=0.01)
h = model.fit(X, Y, epochs=10, validation_data=(X, Y))


# Test du code final sur le boston dataset

In [ ]:
import pandas as pd
import numpy as np

data_url = "http://lib.stat.cmu.edu/datasets/boston"
raw_df = pd.read_csv(data_url, sep="\s+", skiprows=22, header=None)
data = np.hstack([raw_df.values[::2, :], raw_df.values[1::2, :2]])
target = raw_df.values[1::2, 2]
data = data
target = target

X = data
Y = target.reshape((506, 1))

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.25, random_state=0)

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [ ]:
X_train.shape, y_train.shape


In [ ]:
X_test.shape, y_test.shape


## Regression linéaire simple

In [ ]:
model = Model([ Dense(neurons=1)])
mse = Loss()
model.compile(loss=mse, learning_rate=0.001)
h = model.fit(X_train, y_train, epochs=30, validation_data=(X_test, y_test))


In [ ]:
h.keys()


In [ ]:
import matplotlib.pyplot as plt

def plot_learning_curve(history):


  plt.plot(list(range(len(history['loss']))), history['loss'])
  plt.plot(list(range(len(history['val_loss']))), history['val_loss'])
  plt.xlabel('Epochs')
  plt.ylabel("Loss")
  plt.title("Learning Curve")
  plt.show()


In [ ]:
plot_learning_curve(h)


## Réseau de neurones simple

In [ ]:
sigmoid = Sigmoid()
model = Model([ Dense(neurons=3, activation=sigmoid),
                Dense(neurons=1)
                ])


In [ ]:
model


In [ ]:
mse = Loss()
model.compile(loss=mse, learning_rate=0.0001)
h = model.fit(X_train, y_train, epochs=150, validation_data=(X_test, y_test))


In [ ]:
plot_learning_curve(h)


## Deep Neural Network

In [ ]:
sigmoid = Sigmoid()
model = Model([ Dense(neurons=13, activation=sigmoid),
               Dense(neurons=1),

                ])


In [ ]:
model


In [ ]:
mse = Loss()
model.compile(loss=mse, learning_rate=0.0001)
h = model.fit(X_train, y_train, epochs=150, validation_data=(X_test, y_test))


In [ ]:
plot_learning_curve(h)


# Sauvegarder le Modèle

In [ ]:
sigmoid = Sigmoid()
model = Model([ Dense(neurons=13, activation=sigmoid),
               Dense(neurons=1),

                ])


In [ ]:
mse = Loss()
model.compile(loss=mse, learning_rate=0.0001)
h = model.fit(X_train, y_train, epochs=10, validation_data=(X_test, y_test))


In [ ]:
model.save_model('model.file')


In [ ]:
import pickle

def load_model(file):
  with open(file, 'rb') as f:
    model_load = pickle.load(f)
  return model_load


In [ ]:
model_charge = load_model("model.file")
model_charge
